# 16. Decoding Strategies | 大模型解码策略：Top-K, Top-p (Nucleus) 与 Temperature

**难度：** Medium | **标签：** `推理算法`, `Decoding` | **目标人群：** 模型微调与工程部署

有了 `Logits` 之后，我们如何决定大模型生成的下一个词是什么？
这就是解码策略（Decoding Strategy）解决的问题。如果只取概率最大的词（Greedy Search），模型生成的文本会非常干瘪且容易重复。为了让生成的文本既有逻辑性又有创造性，我们需要引入**温度采样（Temperature）**、**Top-K** 和 **Top-p（核采样）**。

在面试大模型算法岗（无论是微调还是部署）时，这三个算法的张量实现是必考的！


### Step 1: 核心思想与痛点

> **大模型的最后一步输出什么？**
> 模型对下一个词的预测是一串未经归一化的得分，称为 `Logits`（形状为 `[vocab_size]`，比如 32000 个数字）。
> 
> **三种常用的截断与平滑策略：**
> 1. **Temperature ($T$)**：在 Softmax 之前，将所有 `Logits` 除以 $T$。
>    - $T < 1$：拉大差距，让高分更高，低分更低（结果更确定）。
>    - $T > 1$：缩小差距，让得分变得平均（结果更随机，也就是更“胡言乱语”）。
> 2. **Top-K 截断**：只保留得分最高的 $K$ 个词的概率，把排名第 $K+1$ 之后的词全部强制剔除（概率置为 $-\infty$）。
> 3. **Top-p (Nucleus) 核采样**：动态截断。按概率从大到小排序，当累加的概率刚好超过阈值 $p$ 时，截断后面的词。它可以根据分布的平缓程度，自动决定截断的数量。


### Step 2: 代码实现框架
在自回归解码中，我们获取最后一个 Token 的概率分布后：
- **Temperature**: 将 Logits 除以 $T$。
- **Top-K**: 用 `torch.topk` 找出前 K 个最大的概率，将其他的 Logits 置为 $-\infty$。
- **Top-P (Nucleus)**: 对 Logits 降序排列并计算累积概率，将累积概率超过 $P$ 的位置对应的 Logits 屏蔽掉。
最后使用 `torch.multinomial` 依概率进行采样。


###  Step 3: 核心机制：Softmax 截断与重整化

**面试必考难点：Top-p 是如何用代码实现的？**
假设词表只有 5 个词，排序后的概率分别是：`[0.5, 0.3, 0.1, 0.05, 0.05]`。阈值 $p = 0.85$。
1. 计算累加和 (Cumulative Sum, `cumsum`)：`[0.5, 0.8, 0.9, 0.95, 1.0]`
2. 找出那些**累加和超过 0.85 的位置**（即 `[False, False, True, True, True]`）
3. 把这些位置对应的原始 Logits 强制设为 `-inf`
4. 对剩下的有效 Logits 重新执行一次 `Softmax` 进行概率归一化。


###  Step 4: 动手实战

**要求**：请依次补全 `apply_temperature`、`apply_top_k` 和 `apply_top_p` 三个函数的核心逻辑，并组合成最终的采样函数。


In [1]:
import torch
import torch.nn.functional as F

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [4]:
a = torch.randn(5, 3)
a[:, -1].unsqueeze(-1)

tensor([[ 0.3864],
        [-2.0689],
        [ 0.3130],
        [-1.7084],
        [-0.7005]])

In [15]:
b = torch.arange(24).view(3, 8)

b = b [..., torch.randperm(b.shape[-1])]
print(b)

tensor([[ 2,  5,  1,  0,  7,  6,  4,  3],
        [10, 13,  9,  8, 15, 14, 12, 11],
        [18, 21, 17, 16, 23, 22, 20, 19]])


In [16]:
val, idx = torch.sort(b, descending=True)

print(val)
print(idx)

tensor([[ 7,  6,  5,  4,  3,  2,  1,  0],
        [15, 14, 13, 12, 11, 10,  9,  8],
        [23, 22, 21, 20, 19, 18, 17, 16]])
tensor([[4, 5, 1, 6, 7, 0, 2, 3],
        [4, 5, 1, 6, 7, 0, 2, 3],
        [4, 5, 1, 6, 7, 0, 2, 3]])


## Scatter (writes)

`tensor.scatter_(dim, index, src)` scatters `src` into `tensor` along `dim`:


>  self[index[i][j]][j] = src[i][j]   # if dim == 0 <br>
>  self[i][index[i][j]] = src[i][j]   # if dim == 1

`index` and `src` must have the same shape (for the value form, only `index` matters). Think of it as the write-counterpart to `torch.gather` (which reads).


The key insight: **`index` does NOT need to match `self`'s shape. It needs to match `src`'s shape** (or be smaller). `scatter_` iterates over **every element of `index`, not over `self`.**


```
  torch.zeros(2, 5).scatter_(1, torch.tensor([[3],[1]]), 1.0)

  Here:
  - self is 2×5
  - index is 2×1
  - value is the scalar 1.0

  The loop runs over every element of index (shape 2×1, so 2 elements):

  for i in range(2):       # index.shape[0]
      for j in range(1):   # index.shape[1]
          self[i][ index[i][j] ] = 1.0    # dim=1, so index replaces the column coordinate

  - i=0, j=0: index[0][0] = 3 → self[0][3] = 1.0
  - i=1, j=0: index[1][0] = 1 → self[1][1] = 1.0
```

  Result:
  [[0, 0, 0, 1, 0],
   [0, 1, 0, 0, 0]]

`5`（`self` 在 scatter 维度上的大小）只是画布够大、能装下最大索引值即可——它是画布尺寸，不是循环边界。</cell id="be34cc78-761d-48ef-b7b3-89f548eae693">

In [18]:
torch.zeros(2, 5).scatter_(1, torch.tensor([[3],[1]]), 1.0)

tensor([[0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0.]])

In [19]:
torch.zeros(2, 5).scatter_(1, torch.tensor([[3]]), 1.0)

tensor([[0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0.]])

## Gather (reads)

`out = src.gather(dim, index)` 沿 `dim` 从 `src` **读取**：

>  out[i][j] = src[index[i][j]][j]   # if dim == 0 <br>
>  out[i][j] = src[i][index[i][j]]   # if dim == 1

和 scatter 一样，它**遍历 `index` 的每个元素**，用 `index` 替换 `dim` 这一维的坐标。区别在于 `index[i][j]` 出现在公式的**右边**（读取来源），所以：

- **输出形状 == `index` 的形状**（不是 `src` 的形状）
- `index` 在 `dim` 以外的维度必须 `<= src` 对应维度

### 一句话对比
| | `gather` | `scatter_` |
|---|---|---|
| 方向 | 从 `src` **读** | 向 `self` **写** |
| `index` 在公式里的位置 | 右侧（来源坐标） | 左侧（目标坐标） |
| 结果形状 | 同 **`index`** | 同 **`self`**（画布） |
| `index` 形状约束 | = 你想要的输出 | = **`src`** |
| 回答的问题 | 「每个槽位，*拉取*哪个元素？」 | 「每个值，*放到*哪里？」 |

```
out  = src.gather(1, idx)      →  out[i][j]            = src[i][ idx[i][j] ]
self.scatter_(1, idx, src)     →  self[i][ idx[i][j] ] = src[i][j]
```

In [24]:
# gather: 输出形状 == index 形状（每行挑 2 列）
src = torch.tensor([[10., 11., 12., 13.],
                    [20., 21., 22., 23.]])
idx = torch.tensor([[3, 0],
                    [1, 2]])
out = src.gather(1, idx)
print(out)
# out[0][0]=src[0][3]=13, out[0][1]=src[0][0]=10
# out[1][0]=src[1][1]=21, out[1][1]=src[1][2]=22
# => [[13, 10],
#     [21, 22]]

tensor([[13., 10.],
        [21., 22.]])


In [25]:
torch.zeros_like(src).scatter_(1, idx, out)

tensor([[10.,  0.,  0., 13.],
        [ 0., 21., 22.,  0.]])

## gather 与 scatter 互为逆操作（Top-p 的核心技巧）

用**同一个 `index`**先 `gather` 再 `scatter_`，能精确回到原始布局。
`torch.sort` 同时返回「排序值」和「索引」，本质上 `sorted_logits == logits.gather(-1, sorted_indices)`；
之后 `scatter_(-1, sorted_indices, ...)` 把它反排序还原。这就是 `apply_top_p` 的全部套路：**排序 → 编辑 → 反排序**。

### `empty_like` vs `zeros_like`
- `zeros_like`：分配 + 填零，读之前安全。**部分写入**（如上面的 one-hot）时必须用，否则没写到的位置是垃圾值。
- `empty_like`：只分配、不填零，更快。**仅当保证每个元素都会被覆盖**时才用。
- 在 `apply_top_p` 里 `sorted_indices` 是完整排列，会写满每个位置，所以 `empty_like` 也安全且更快（`zeros_like` 在这里属于多余的填零）。

In [ ]:
# gather 与 scatter 互为逆：sort → (edit) → 用同一 index 还原
logits = torch.tensor([[0.1, 2.3, 0.4, 1.2, -0.5]])
print('original     :', logits.tolist())

# sort 本质就是按某个排列做 gather
sorted_logits, idx = torch.sort(logits, descending=True)
print('sorted_logits:', sorted_logits.tolist())
print('idx          :', idx.tolist())
print('gather==sort :', torch.equal(sorted_logits, logits.gather(-1, idx)))

# scatter_ 用同一个 idx 把它反排序还原（完整排列 → empty_like 即可，会写满每个位置）
restored = torch.empty_like(logits).scatter_(-1, idx, sorted_logits)
print('restored     :', restored.tolist())
print('round-trip ok:', torch.equal(restored, logits))

In [20]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """
    应用温度调节。注意：通常 T=1.0 意味着不改变，T 越接近 0 越确定（Greedy）。
    """
    # ==========================================
    # TODO 1: 确保 temperature 至少大于一个极小值(如 1e-6) 防止除零
    # 然后让 logits 统一除以 temperature
    # ==========================================
    temp = max(temperature, 1e-6)
    return logits / temperature

def apply_top_k(logits: torch.Tensor, top_k: int) -> torch.Tensor:
    """
    Top-K 截断。只保留值最大的 top_k 个，其余置为 -inf。
    """
    # 如果 top_k 是 0 或超过词表大小，不处理
    if top_k <= 0 or top_k >= logits.size(-1):
        return logits
        
    # ==========================================
    # TODO 2: 实现 Top-K 截断
    # ==========================================
    filter_value = float('-inf')

    k_values, _ = torch.topk(logits, top_k, dim=-1, largest=True, sorted=True)
    kth_value = k_values[:, -1].unsqueeze(-1)
    
    logits = torch.where(logits < kth_value, torch.tensor(filter_value, device=logits.device), logits)
    return logits

def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """
    Top-p (Nucleus) 核采样截断。
    """
    if top_p <= 0.0 or top_p >= 1.0:
        return logits
        
    # 1. 首先需要将 logits 从大到小排序
    # 注意我们需要记住原始的索引 (indices)，因为截断完了还要把它复原回原来的位置！
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    
    # 2. 对排序后的概率 (需要先算一遍 Softmax) 计算累加和 (Cumulative Probability)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    # ==========================================
    # TODO 3: 实现 Top-P 核心逻辑
    # 1. 找出 cumulative_probs > top_p 的所有位置。这就是我们需要剔除(丢弃)的 token
    # 2. 我们想保留第一个累加概率 > p 的词，所以需要把这个掩码向右平移一位
    # 3. 将这些被剔除位置对应的 sorted_logits 设为 -inf
    # 4. 把剔除后的 sorted_logits 按照 sorted_indices 散布 (scatter) 回原来的形状里
    # ==========================================
    
    # 找到需要丢弃的掩码
    sorted_indices_to_remove = cumulative_probs > top_p
    
    # 向右平移掩码以保留最后一个刚好超过阈值的 token
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0  # 确保无论如何最高概率的 token 不被丢弃
    
    # 将需要剔除的 sorted_logits 设为极小值
    sorted_logits[sorted_indices_to_remove] = float('-inf')
    restored_logits = torch.empty_like(logits).scatter_(
        dim=-1, index=sorted_indices, src=sorted_logits
    )
    return restored_logits


def decode_next_token(logits: torch.Tensor, temperature=0.7, top_k=50, top_p=0.9):
    """
    组合以上三种策略，并通过 Multinomial 进行随机多项式采样
    """
    # 1. 调温
    logits = apply_temperature(logits, temperature)
    
    # 2. Top-K 截断 (通常先 K 后 p)
    logits = apply_top_k(logits, top_k)
    
    # 3. Top-p 截断
    logits = apply_top_p(logits, top_p)
    
    # 4. 概率重归一化
    probs = F.softmax(logits, dim=-1)
    
    # 5. 从概率分布中采样 1 个词
    next_token = torch.multinomial(probs, num_samples=1)
    
    return next_token


In [21]:
# 运行此单元格以测试你的实现
def test_decoding():
    try:
        # 为了保证可重复性
        torch.manual_seed(42)
        vocab_size = 10
        # 伪造一组 Logits: [0.1, 2.3, 0.4, 1.2, -0.5, 4.0, 3.1, 0.0, 1.1, -1.0]
        # 最大的两个: index 5 (4.0), index 6 (3.1)
        logits = torch.tensor([[0.1, 2.3, 0.4, 1.2, -0.5, 4.0, 3.1, 0.0, 1.1, -1.0]])
        
        print("原始 Logits (前10个单词):", logits.squeeze().tolist())
        
        # 1. 测试 Temperature
        t_logits = apply_temperature(logits.clone(), 0.5)
        # 温度 0.5 应该会让差异翻倍
        assert torch.allclose(t_logits[0, 5] - t_logits[0, 6], (logits[0, 5] - logits[0, 6]) * 2), "温度调节错误！"
        print("✅ Temperature 热量调节通过！")
        
        # 2. 测试 Top-K
        k_logits = apply_top_k(logits.clone(), 3)
        # 只保留最大的三个：5, 6, 1
        valid_count = (k_logits != float('-inf')).sum().item()
        assert valid_count == 3, f"Top-K 截断没有正确执行，保留了 {valid_count} 个值"
        print("✅ Top-K 暴力截断通过！")
        
        # 3. 测试 Top-p
        # 原始概率: [0.01, 0.10, 0.01, 0.03, 0.00, 0.54, 0.22, 0.01, 0.03, 0.00]
        # 降序: 0.54 (idx 5), 0.22 (idx 6), 0.10 (idx 1) ...
        # 累加和: 0.54, 0.76, 0.86
        # 所以只有 idx 5, 6, 1 会被保留
        p_logits = apply_top_p(logits.clone(), 0.8)
        valid_count = (p_logits != float('-inf')).sum().item()
        assert valid_count == 3, f"Top-p 核采样截断没算准，保留了 {valid_count} 个值"
        print("✅ Top-p (Nucleus) 核采样动态截断通过！")
        
        # 4. 测试完整管线
        next_token = decode_next_token(logits.clone(), temperature=0.7, top_k=50, top_p=0.9)
        assert next_token.shape == (1, 1), "解码的词张量维度不对"
        print(f"\n✅ All Tests Passed! 解码策略实现通过测试。本次采样的下一个词汇 ID 是: {next_token.item()}")
        
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
    except TypeError as e:
        print("代码可能未完成，导致了操作错误。")
    except Exception as e:
        print(f"❌ 发生未知异常: {e}")

test_decoding()


原始 Logits (前10个单词): [0.10000000149011612, 2.299999952316284, 0.4000000059604645, 1.2000000476837158, -0.5, 4.0, 3.0999999046325684, 0.0, 1.100000023841858, -1.0]
✅ Temperature 热量调节通过！
✅ Top-K 暴力截断通过！
✅ Top-p (Nucleus) 核采样动态截断通过！

✅ All Tests Passed! 解码策略实现通过测试。本次采样的下一个词汇 ID 是: 5


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """
    应用温度调节。注意：通常 T=1.0 意味着不改变，T 越接近 0 越确定（Greedy）。
    """
    # TODO 1: 确保 temperature 至少大于一个极小值(如 1e-6) 防止除零
    temp = max(temperature, 1e-6)
    return logits / temp

def apply_top_k(logits: torch.Tensor, top_k: int) -> torch.Tensor:
    """
    Top-K 截断。只保留值最大的 top_k 个，其余置为 -inf。
    """
    if top_k <= 0 or top_k >= logits.size(-1):
        return logits
        
    # TODO 2: 实现 Top-K 截断
    filter_value = float('-inf')
    
    # 找到第 K 大的值
    kth_values, _ = torch.topk(logits, top_k, dim=-1, largest=True, sorted=True)
    kth_values = kth_values[..., -1:] # 取最后一个（第 K 大的值）
    
    # 将小于第 K 大值的位置设为 -inf
    logits = torch.where(logits < kth_values, torch.tensor(filter_value, device=logits.device), logits)
    return logits

def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """
    Top-p (Nucleus) 核采样截断。
    """
    if top_p <= 0.0 or top_p >= 1.0:
        return logits
        
    # 1. 首先需要将 logits 从大到小排序
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    
    # 2. 对排序后的概率计算累加和
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    # TODO 3: 实现 Top-P 核心逻辑
    # 找到需要丢弃的掩码
    sorted_indices_to_remove = cumulative_probs > top_p
    
    # 向右平移掩码以保留最后一个刚好超过阈值的 token
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0  # 确保无论如何最高概率的 token 不被丢弃
    
    # 将需要剔除的 sorted_logits 设为极小值
    sorted_logits[sorted_indices_to_remove] = float('-inf')
    
    # 将排序后的 logits 恢复到原始顺序
    restored_logits = torch.zeros_like(logits).scatter_(
        dim=-1, index=sorted_indices, src=sorted_logits
    )
    
    return restored_logits

def decode_next_token(logits: torch.Tensor, temperature=0.7, top_k=50, top_p=0.9):
    """
    组合以上三种策略，并通过 Multinomial 进行随机多项式采样
    """
    # 1. 调温
    logits = apply_temperature(logits, temperature)
    
    # 2. Top-K 截断 (通常先 K 后 p)
    logits = apply_top_k(logits, top_k)
    
    # 3. Top-p 截断
    logits = apply_top_p(logits, top_p)
    
    # 4. 概率重归一化
    probs = F.softmax(logits, dim=-1)
    
    # 5. 从概率分布中采样 1 个词
    next_token = torch.multinomial(probs, num_samples=1)
    
    return next_token

### 解析

**1. TODO 1: Temperature 温度调节**
- **实现方式**：`temp = max(temperature, 1e-6)`，`return logits / temp`
- **关键点**：温度越低（T < 1），分布越尖锐，模型越确定；温度越高（T > 1），分布越平缓，模型越随机
- **技术细节**：防止除零错误，确保 temperature 至少为 1e-6

**2. TODO 2: Top-K 截断**
- **实现方式**：`kth_values = torch.topk(logits, top_k)[0][..., -1:]`，`logits = torch.where(logits < kth_values, -inf, logits)`
- **关键点**：只保留概率最高的 K 个词，其余词的 logits 设为负无穷
- **技术细节**：使用 `torch.topk` 找到第 K 大的值，然后用 `torch.where` 进行条件替换

**3. TODO 3: Top-p (Nucleus) 核采样**
- **实现方式**：对 logits 降序排序，计算累积概率，找到累积概率超过 p 的位置并屏蔽，最后用 `scatter_` 恢复原始顺序
- **关键点**：动态截断——根据概率分布的形状自动决定保留多少个词
- **技术细节**：
  - 向右平移掩码（`sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()`）确保保留第一个超过阈值的词
  - 使用 `scatter_` 将排序后的 logits 恢复到原始索引顺序

**工程优化要点**
- **Temperature 先行**：温度调节应该在截断之前进行，因为它影响概率分布的形状
- **Top-K vs Top-p**：Top-K 是固定数量截断，Top-p 是动态截断，通常先应用 Top-K 再应用 Top-p
- **数值稳定性**：使用 `-float('inf')` 而非直接删除元素，保持张量形状不变
- **采样策略**：`torch.multinomial` 根据归一化后的概率分布进行多项式采样
- **工业实践**：不同任务需要不同的超参数组合——代码生成通常用低温度（0.2-0.5），创意写作用高温度（0.7-1.0）